# NHL play-by-play data acquisition

This notebook explains how to use the team's `NHLDataRetriever` to discover completed NHL games, download raw play-by-play JSON, and keep a local cache for later analysis. It covers the required seasons from **2016–17 through 2023–24**, including regular season and playoffs.

## Before you run

- From the project root, install the environment with `uv sync --locked`, then start Jupyter with `uv run jupyter lab`. See the [project setup instructions](../README.md#setup).
- If you want a custom local data path, copy `.env.example` to `.env` at the project root and edit `NHL_DATA_DIR`. The default is `data/raw/nhl`. The `.env` file and downloaded data are ignored by Git; do not commit them.
- The setup/import cell below makes **no NHL requests**. The example game and small pilot cells are off by default. The schedule-count cell is opt-in and retrieves only game-list metadata (at most one request per season when not cached).
- The final full-acquisition cell is also **off by default**. An initial full run can make thousands of play-by-play requests and take hours. The retriever spaces uncached requests by at least one second, saves each game separately, and reuses completed cache files on reruns. Run one explicit step at a time; do not use “Run All” without reviewing the opt-in flags.

## 1. Configure the retriever

This imports the implementation from the project package, finds the repository root whether Jupyter was started there or in `notebooks/`, and loads the local `.env`. Creating the retriever does not create directories or contact the NHL.

In [1]:
from pathlib import Path

from dotenv import load_dotenv
from ift6758.data.question_1 import NHLDataRetriever

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "ift6758" / "data" / "question_1.py").is_file():
    raise RuntimeError(
        "Open this notebook from the team project root or its notebooks folder."
    )

load_dotenv(PROJECT_ROOT / ".env", override=False)
retriever = NHLDataRetriever(project_root=PROJECT_ROOT)

print(f"Project: {PROJECT_ROOT}")
print(f"Local data directory: {retriever.data_dir}")
print(f"Minimum interval between requests: {retriever.min_request_interval} seconds")

Project: D:\School\IFT6758_Data_Science\workspace\project
Local data directory: D:\School\IFT6758_Data_Science\workspace\project\data\raw\nhl
Minimum interval between requests: 1.0 seconds


The required starting years are 2016 through 2023. The retriever discovers real completed game IDs from the NHL schedule instead of assuming every possible ID exists. Game type `02` means regular season and `03` means playoffs. For example, `2023030111` is a 2023–24 playoff game.

In [2]:
SEASON_START_YEARS = list(range(2016, 2024))
SEASON_START_YEARS

[2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

## 2. Optional one-game example

Set the flag to `True` and run this cell to fetch or load one known playoff game. If the game is not already cached, this makes one play-by-play request. The raw response is saved as `NHL_DATA_DIR/games/2023-24/<game_id>.json`; running the cell again uses that file without another request.

In [3]:
RUN_SINGLE_GAME_EXAMPLE = False
EXAMPLE_GAME_ID = 2023030111

if RUN_SINGLE_GAME_EXAMPLE:
    game = retriever.get_game(EXAMPLE_GAME_ID)
    print(f"Game ID: {game['id']}")
    print(f"Play-by-play events: {len(game['plays'])}")
else:
    print("No request made. Set RUN_SINGLE_GAME_EXAMPLE = True to run the example.")

No request made. Set RUN_SINGLE_GAME_EXAMPLE = True to run the example.


## 3. Optional schedule check

This discovers the number of completed regular-season and playoff games in each required season. It downloads **schedule metadata only**, not play-by-play files. On a fresh machine it can make up to eight metadata requests; subsequent runs use the local metadata cache. Checking these counts before the large download helps confirm what the retriever plans to acquire.

In [ ]:
SHOW_SEASON_COUNTS = False

if SHOW_SEASON_COUNTS:
    season_game_ids = {}
    for start_year in SEASON_START_YEARS:
        game_ids = retriever.get_season_game_ids(start_year)
        season_game_ids[start_year] = game_ids
        season = f"{start_year}-{str(start_year + 1)[-2:]}"
        print(f"{season}: {len(game_ids)} completed regular-season/playoff games")
else:
    print("No schedule requests made. Set SHOW_SEASON_COUNTS = True to inspect seasons.")

2016-17: 1317 completed regular-season/playoff games
2017-18: 1355 completed regular-season/playoff games
2018-19: 1358 completed regular-season/playoff games
2019-20: 1212 completed regular-season/playoff games
2020-21: 952 completed regular-season/playoff games
2021-22: 1401 completed regular-season/playoff games
2022-23: 1400 completed regular-season/playoff games
2023-24: 1400 completed regular-season/playoff games


## 4. Optional small pilot

Before acquiring a full season, you can try at most two games from 2023–24. This also checks that the schedule discovery and play-by-play cache work together. A cached game is reused; only missing files require requests.

In [ ]:
RUN_SMALL_PILOT = False
PILOT_SEASON_START_YEAR = 2023
PILOT_MAX_GAMES = 2

if RUN_SMALL_PILOT:
    pilot_report = retriever.get_season(
        PILOT_SEASON_START_YEAR, max_games=PILOT_MAX_GAMES
    )
    print(
        f"Season {pilot_report['season']}: "
        f"{len(pilot_report['succeeded'])}/{len(pilot_report['selected'])} succeeded; "
        f"cached={pilot_report['cached']}, "
        f"downloaded={pilot_report['downloaded']}, "
        f"failed={len(pilot_report['failed'])}"
    )
else:
    print("No games requested. Set RUN_SMALL_PILOT = True to retrieve at most two.")

Season 20232024: 2/2 succeeded; cached=2, downloaded=0, failed=0


## 5. Acquire all required seasons (explicit opt-in)

Only set `CONFIRM_FULL_ACQUISITION` to `True` when you are ready to start the full download. This calls `get_season` once for each starting year, 2016–2023. It discovers the actual completed game IDs, saves each raw game separately, records failures, and counts cache hits.

If the process is interrupted, rerun this cell: already-saved games are loaded from cache, so only missing or previously unsuccessful games need another request. The report is for this run; inspect failures and rerun after resolving transient problems. Do not commit the `data/` directory.

In [ ]:
CONFIRM_FULL_ACQUISITION = True
full_reports = {}

if CONFIRM_FULL_ACQUISITION:
    for start_year in SEASON_START_YEARS:
        report = retriever.get_season(start_year)
        full_reports[start_year] = report
        print(
            f"{report['season']}: "
            f"{len(report['succeeded'])}/{len(report['selected'])} succeeded; "
            f"cached={report['cached']}, "
            f"downloaded={report['downloaded']}, "
            f"failed={len(report['failed'])}"
        )
        if report["failed"]:
            print("  First failures:", list(report["failed"].items())[:5])
else:
    print("Full acquisition not started. Review the warning, set the flag to True, and run this cell intentionally.")

## 6. Check the run report

This check uses the in-memory reports from the full-acquisition cell and makes no network requests. A season is complete only when every selected game succeeded, no failures were recorded, and all expected JSON files are present. If a season is missing from the report, the full-acquisition cell has not finished it in this kernel.

In [ ]:
if not full_reports:
    print("No full-acquisition report is available in this kernel yet.")
else:
    not_run = [year for year in SEASON_START_YEARS if year not in full_reports]
    if not_run:
        print("Seasons not completed in this run:", not_run)

    for start_year, report in full_reports.items():
        selected_ids = set(report["selected"])
        succeeded_ids = set(report["succeeded"])
        missing_ids = sorted(selected_ids - succeeded_ids)
        season_folder = f"{start_year}-{str(start_year + 1)[-2:]}"
        missing_files = [
            game_id
            for game_id in sorted(selected_ids)
            if not (retriever.data_dir / "games" / season_folder / f"{game_id}.json").is_file()
        ]
        complete = (
            not report["failed"]
            and len(succeeded_ids) == len(selected_ids)
            and not missing_files
        )
        print(
            f"{report['season']}: complete={complete}; "
            f"missing games={len(missing_ids)}; missing files={len(missing_files)}"
        )
        if missing_ids:
            print("  First missing game IDs:", missing_ids[:10])

## What the reports mean

- `discovered`: completed regular-season and playoff game IDs in the NHL schedule metadata.
- `selected`: IDs chosen for this call (all discovered games for the full-season cell).
- `succeeded`: IDs whose game JSON was loaded or fetched and passed the retriever's basic checks.
- `cached` / `downloaded`: successful games reused locally / fetched during this call.
- `failed`: IDs that could not be retrieved or loaded, with an error message. Review these before claiming full coverage.

The raw payloads remain on disk rather than accumulating in a large in-memory collection. Once acquisition is complete, later analysis notebooks can read the local JSON files without repeating NHL requests.